## 스쿨존 안전점수 산출 기준

| 구분 | 컬럼 | 가중치 | 산출 방식 | 근거 |
|------|------|--------|------------|------|
| 위험(감산) | 발생건수 | 30% | 정규화 후 감산 | 가장 직접적인 사고 이력 지표 |
| 위험(감산) | 생활안전CCTV | 6% | 수량 정규화 후 감산 | 차량 통행이 많은 구역의 프록시 |
| 위험(감산) | 무인교통단속카메라 | 5% | 수량 정규화 후 감산 | 차량 밀도 높은 구역 암시 |
| 안전(가산) | 도로적색표면 | 13% | 유무(0/1) 또는 면적 정규화 후 가산 | 시인성 확보, 핵심 안전시설 |
| 안전(가산) | 신호등 | 11% | 수량 정규화 후 가산 | 차량·보행자 충돌 방지 핵심 시설 |
| 안전(가산) | 횡단보도 | 7% | 유무(0/1) 후 가산 | 보행자 안전 보장 기본 시설 |
| 안전(가산) | 도로안전표지 | 7% | 수량 정규화 후 가산 | 경고·규제 기능 기본 시설 |
| 안전(가산) | 보호구역표지판 | 7% | 유무(0/1) 후 가산 | 스쿨존 인식 제고 |
| 안전(가산) | 무단횡단방지펜스 | 7% | 수량 정규화 후 가산 | 직접적 무단횡단 방지 시설 |
| 안전(가산) | 옐로카펫 | 5% | 유무(0/1) 후 가산 | 보행자 대기 안전 보조 시설 |
| 안전(가산) | 어린이비율 | 2% | 정규화 후 가산 | 발생건수와 음의 상관관계(-0.31), 스쿨존 효과 반영 |

In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'C:\Users\EL36\Desktop\1차프로젝트_CCTV\1stProject_MSAI09\Preprocessing\5_seongnam_final_dataset.csv')

# 복사본 사용
data = df.copy()

# -------------------------
# 1. 정규화 함수 정의
# -------------------------
def min_max_normalize(series):
    return (series - series.min()) / (series.max() - series.min() + 1e-9)

# -------------------------
# 2. 정규화 대상 컬럼
# -------------------------

normalize_cols = [
    '발생건수',
    '생활안전CCTV',
    '무인교통단속카메라',
    '신호등',
    '도로안전표지',
    '무단횡단방지펜스',
    '어린이 비율(%)'
]

for col in normalize_cols:
    data[col + '_norm'] = min_max_normalize(data[col])

# 유무(0/1) 변수는 그대로 사용
binary_cols = [
    '도로적색표면',
    '횡단보도',
    '보호구역표지판',
    '옐로카펫'
]

# -------------------------
# 3. 가중치 정의
# -------------------------

weights = {
    '발생건수': 0.30,
    '생활안전CCTV': 0.06,
    '무인교통단속카메라': 0.05,
    '도로적색표면': 0.13,
    '신호등': 0.11,
    '횡단보도': 0.07,
    '도로안전표지': 0.07,
    '보호구역표지판': 0.07,
    '무단횡단방지펜스': 0.07,
    '옐로카펫': 0.05,
    '어린이 비율(%)': 0.02
}

# -------------------------
# 4. 점수 계산
# -------------------------

# 위험(감산)
risk_score = (
    weights['발생건수'] * data['발생건수_norm'] +
    weights['생활안전CCTV'] * data['생활안전CCTV_norm'] +
    weights['무인교통단속카메라'] * data['무인교통단속카메라_norm']
)

# 안전(가산)
safety_score = (
    weights['도로적색표면'] * data['도로적색표면'] +
    weights['신호등'] * data['신호등_norm'] +
    weights['횡단보도'] * data['횡단보도'] +
    weights['도로안전표지'] * data['도로안전표지_norm'] +
    weights['보호구역표지판'] * data['보호구역표지판'] +
    weights['무단횡단방지펜스'] * data['무단횡단방지펜스_norm'] +
    weights['옐로카펫'] * data['옐로카펫'] +
    weights['어린이 비율(%)'] * data['어린이 비율(%)_norm']
)

# -------------------------
# 5. 최종 안전점수 산출
# -------------------------

data['안전점수_raw'] = safety_score - risk_score

# 0~100 스케일링
data['안전점수'] = min_max_normalize(data['안전점수_raw']) * 100

data.head(3)

,시설물명,위도,경도,도로안전표지,도로적색표면,무단횡단방지펜스,무인교통단속카메라,보호구역표지판,생활안전CCTV,신호등,...,어린이 비율(%),발생건수_norm,생활안전CCTV_norm,무인교통단속카메라_norm,신호등_norm,도로안전표지_norm,무단횡단방지펜스_norm,어린이 비율(%)_norm,안전점수_raw,안전점수
0,중원초등학교,37.437855,127.167857,0,5,12,0,15,18,0,...,6.482880,0.0,0.246154,0.0,0.000000,0.000000,0.266667,0.148936,1.756876,16.171406
1,하원초등학교,37.446255,127.170236,6,7,21,2,15,16,10,...,10.261116,0.0,0.215385,0.2,0.588235,0.461538,0.466667,0.442203,2.595601,25.611370
2,중부초등학교,37.451546,127.164951,2,19,25,2,26,27,4,...,6.042885,0.0,0.384615,0.2,0.235294,0.153846,0.555556,0.114783,4.524759,47.324305


#### 위 계산식은 '사고여부'를 포함하고 있어 순환논리 오류가 발생하므로 폐기

### 신규 코드 제작

In [4]:
# 모델링을 위한 데이터준비
import pandas as pd

# 이미지 확률 데이터
class_df = pd.read_csv(r"C:\Users\EL36\Desktop\1차프로젝트_CCTV\1stProject_MSAI09\CustomVision\0_classification_features.csv")
class_df["시설물명"] = class_df["image"].str.replace("_북쪽.png", "", regex=False)
class_df["시설물명"] = class_df["시설물명"].str.replace(r"_\d+", "", regex=True)
class_df.head(3)

,image,p_wide,p_narrow,p_barrier_yes,p_barrier_no,시설물명
0,AMI몬테소리손바닥어린이집_북쪽.png,0.660075,0.344503,0.382240,0.614802,AMI몬테소리손바닥어린이집
1,AP어린이집_북쪽.png,0.940672,0.060079,0.822805,0.178012,AP어린이집
2,EPS어린이집_북쪽.png,0.004515,0.995615,0.005522,0.994376,EPS어린이집


In [5]:

obj_df = pd.read_csv(r"C:\Users\EL36\Desktop\1차프로젝트_CCTV\1stProject_MSAI09\CustomVision\0_object_detection_features.csv")

obj_df["시설물명"] = obj_df["image"].str.replace("_북쪽.png", "", regex=False)
obj_df["시설물명"] = obj_df["시설물명"].str.replace(r"_\d+", "", regex=True)
obj_df.head(3)

,image,road_width_relative,sidewalk_ratio,parked_density,시설물명
0,AMI몬테소리손바닥어린이집_북쪽.png,0.400759,0.0,1,AMI몬테소리손바닥어린이집
1,AP어린이집_북쪽.png,0.428197,0.0,2,AP어린이집
2,EPS어린이집_북쪽.png,0.000000,0.0,0,EPS어린이집


In [6]:
# 커스텀비전으로 만든 img feature끼리 먼저 병합
image_df = class_df.merge(obj_df, on=["image", "시설물명"], how="inner")

In [ ]:
image_df = class_df.merge(obj_df, on=["image", "시설물명"], how="inner")
image_df.to_csv(r"C:\Users\EL36\Desktop\1차프로젝트_CCTV\1stProject_MSAI09\CustomVision\accidentlevel.csv", index=False)

In [3]:
# 전국 이미지 기반 데이터
image_df = pd.read_csv(r"C:\Users\EL36\Desktop\1차프로젝트_CCTV\1stProject_MSAI09\CustomVision\accidentlevel.csv")

# 라벨 생성
image_df["accident_label"] = image_df["image"].apply(
    lambda x: 1 if "부근" in x else 0
)

In [4]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

df = image_df.copy()

features = [
    "p_wide",
    "p_barrier_yes",
    "road_width_relative",
    "sidewalk_ratio",
    "parked_density"
]

X = df[features]
y = df["accident_label"]

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(max_iter=1000))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_auc = cross_val_score(
    model,
    X,
    y,
    cv=skf,
    scoring="roc_auc"
)

print("CV AUC:", cv_auc)
print("Mean CV AUC:", cv_auc.mean())

CV AUC: [0.70527168 0.69661802 0.71253386 0.72316782 0.71023899]
Mean CV AUC: 0.7095660736792702


In [5]:
#학습
model.fit(X, y)
df["structure_risk"] = model.predict_proba(X)[:, 1]

In [6]:
# 전국 데이터 기준 위험확률
model.fit(X, y)
df["structure_risk"] = model.predict_proba(X)[:, 1]


In [7]:
# structure_risk 분포확인
df.groupby("accident_label")["structure_risk"].mean()

accident_label
0    0.373546
1    0.508102
Name: structure_risk, dtype: float64

In [23]:
# 성남시 스쿨존 데이터에서 구조 위험 확률 계산
seongnam_df = pd.read_csv(r"C:\Users\EL36\Desktop\1차프로젝트_CCTV\1stProject_MSAI09\Preprocessing\5_seongnam_final_dataset.csv")
facility_df = seongnam_df.merge(image_df, on="시설물명", how="inner")
facility_df["accident_label"] = (facility_df["발생건수"] > 1).astype(int)

In [24]:
facility_df["structure_risk"] = model.predict_proba(
    facility_df[features]
)[:, 1]

In [25]:
# 성남시 스쿨존 데이터에서 구조 위험의 실제 설명력 확인
# 정상이라면: 사고지역 평균 > 일반지역 평균
facility_df.groupby("accident_label")["structure_risk"].mean()

accident_label
0    0.376339
1    0.326939
Name: structure_risk, dtype: float64

In [26]:
# 사고 예측 모델 구축하기

features_final = [
    "structure_risk",
    "도로안전표지",
    "도로적색표면",
    "무단횡단방지펜스",
    "무인교통단속카메라",
    "보호구역표지판",
    "생활안전CCTV",
    "신호등",
    "옐로카펫",
    "횡단보도",
    "어린이 비율(%)"
]

X = facility_df[features_final]
y = facility_df["accident_label"]

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

model2 = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_auc2 = cross_val_score(
    model2,
    X,
    y,
    cv=skf,
    scoring="roc_auc"
)

print("Policy Model Mean AUC:", cv_auc2.mean())

Policy Model Mean AUC: 0.8140476190476191


In [27]:
from sklearn.metrics import roc_auc_score

roc_auc_score(
    facility_df["accident_label"],
    facility_df["structure_risk"]
)

0.3942307692307693

In [ ]:
#중요 변수 확인
model2.fit(X, y)

import pandas as pd

coef = pd.Series(
    model2.named_steps["logit"].coef_[0],
    index=features_final
).sort_values(ascending=False)

print(coef)

보호구역표지판           2.094643
생활안전CCTV          1.461748
structure_risk    1.047951
무인교통단속카메라         0.663877
어린이 비율(%)         0.220581
무단횡단방지펜스          0.210429
옐로카펫             -0.037805
도로안전표지           -0.077894
신호등              -0.540597
횡단보도             -0.542413
도로적색표면           -2.251642
dtype: float64


① structure risk _ 구조 위험은 독립적으로 사고와 연결됨

② 일부 시설은 “사고 후 설치” 효과를 반영 (보호구역 표지판, 생활안전cctv, 무인교통단속카메라, 펜스)

③ 일부 시설은 실제 완화 효과를 보임 (도로 적색표면, 신호등, 횡단보도)